# Stage 2: SBERT Embeddings Generator for Kaggle
## Process Stage 1 Files → Generate w3 and w5 Embeddings → Save to Stage 2

**📥 Input:** Upload Stage 1 CSV files as Kaggle dataset named `stage-1`
- Expected location: `/kaggle/input/stage-1/`
- Files should be named: `Data_s1_1.csv`, `Data_s1_2.csv`, etc.

**📤 Output:** Stage 2 CSV files with embeddings
- Saved to: `/kaggle/working/stage_2/`
- Files will be named: `Data_s2_1.csv`, `Data_s2_2.csv`, etc.

**Features:**
- ✅ **GPU/CPU auto-detection** (automatically uses GPU if available)
- ✅ Processes files one by one (memory efficient)
- ✅ Dual window embeddings (w3 and w5)
- ✅ Progress tracking and error handling
- ✅ Batch processing optimized for device type
- ✅ Works with Kaggle dataset input system

**🚀 How to Use:**
1. Create a Kaggle dataset with your Stage 1 files
2. Name the dataset: `stage-1` (or update the path below)
3. Add the dataset to this notebook's input
4. Run all cells
5. Download results from `/kaggle/working/stage_2/`

## Step 1: Install Dependencies and Check Environment

In [ ]:
# Check if running on Kaggle
import os
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("✅ Running on Kaggle environment")
    print(f"   Available input datasets:")
    if os.path.exists('/kaggle/input'):
        for dataset in os.listdir('/kaggle/input'):
            print(f"   - {dataset}")
else:
    print("⚠️  Not running on Kaggle - using local paths")

In [ ]:
# Install required packages
!pip install -q sentence-transformers pandas numpy

print("✅ All packages installed!")

In [ ]:
# Import required libraries
import os
import time
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

# Check GPU availability
if torch.cuda.is_available():
    print(f"🎮 GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("💻 No GPU detected - will use CPU")

## Step 2: Configure Paths and Parameters

In [ ]:
# ============================================================================
# KAGGLE PATHS CONFIGURATION
# ============================================================================
# IMPORTANT: Update dataset name if different

# Kaggle dataset input path
# Expected: Upload your Stage 1 files as a Kaggle dataset named "stage-1"
KAGGLE_INPUT_DATASET = 'stage-1'  # Change this to match your dataset name
STAGE_1_INPUT = Path(f'/kaggle/input/{KAGGLE_INPUT_DATASET}')

# Output: Stage 2 folder in Kaggle working directory
# All files here will be available for download after completion
STAGE_2_OUTPUT = Path('/kaggle/working/stage_2')

# Create output directory
STAGE_2_OUTPUT.mkdir(parents=True, exist_ok=True)

print("📁 Kaggle Directory Structure:")
print(f"   Stage 1 Input:  {STAGE_1_INPUT}")
print(f"   Stage 2 Output: {STAGE_2_OUTPUT}")
print(f"\n💡 Output files can be downloaded from the 'Output' tab after completion")

# Verify Stage 1 folder exists
if not STAGE_1_INPUT.exists():
    print(f"\n❌ ERROR: Stage 1 input not found at {STAGE_1_INPUT}")
    print(f"\n📋 How to fix:")
    print(f"   1. Create a Kaggle dataset with your Stage 1 CSV files")
    print(f"   2. Name it '{KAGGLE_INPUT_DATASET}' (or update KAGGLE_INPUT_DATASET variable)")
    print(f"   3. Add the dataset to this notebook:")
    print(f"      - Click 'Add Data' → 'Your Datasets'")
    print(f"      - Select your Stage 1 dataset")
    print(f"   4. Re-run this cell")
    
    # List available datasets
    print(f"\n📦 Currently available input datasets:")
    input_dir = Path('/kaggle/input')
    if input_dir.exists():
        datasets = list(input_dir.iterdir())
        if datasets:
            for ds in datasets:
                print(f"   - {ds.name}")
        else:
            print(f"   (No datasets attached)")
else:
    stage1_files = list(STAGE_1_INPUT.glob('*.csv'))
    print(f"\n✅ Found {len(stage1_files)} CSV files in Stage 1 dataset")
    
    if len(stage1_files) > 0:
        print(f"\n📋 Sample files:")
        for i, f in enumerate(stage1_files[:5], 1):
            file_size_mb = f.stat().st_size / (1024 * 1024)
            print(f"   {i}. {f.name} ({file_size_mb:.2f} MB)")
        if len(stage1_files) > 5:
            print(f"   ... and {len(stage1_files) - 5} more files")

In [ ]:
# ============================================================================
# PROCESSING PARAMETERS - AUTO GPU/CPU DETECTION
# ============================================================================

# Automatically detect and configure for GPU or CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Adjust batch size based on device
# GPU can handle larger batches for faster processing
if device == 'cuda':
    batch_size = 128  # Larger batch for GPU
    print("🎮 GPU Mode: Using optimized settings for GPU acceleration")
else:
    batch_size = 32   # Smaller batch for CPU
    print("💻 CPU Mode: Using CPU-optimized settings")

# Stage 2 Configuration
STAGE_2_CONFIG = {
    'model_name': 'all-mpnet-base-v2',
    'embedding_dim': 768,
    'batch_size': batch_size,
    'device': device,
    'show_progress': True
}

print("\n⚙️  Configuration:")
print(f"   Model: {STAGE_2_CONFIG['model_name']}")
print(f"   Device: {STAGE_2_CONFIG['device'].upper()} {'🚀' if device == 'cuda' else '🐢'}")
print(f"   Batch Size: {STAGE_2_CONFIG['batch_size']} (optimized for {device.upper()})")
print(f"   Embedding Dimension: {STAGE_2_CONFIG['embedding_dim']}")

if device == 'cuda':
    print(f"\n💡 Tip: GPU acceleration is enabled! Processing will be much faster.")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print(f"\n💡 Tip: To use GPU, go to Runtime → Change runtime type → Select GPU")

print("\n✅ Configuration loaded!")

## Step 3: Load SBERT Model

In [ ]:
# ============================================================================
# LOAD SBERT MODEL WITH GPU/CPU SUPPORT
# ============================================================================

print("🔄 Loading SBERT model...")
print(f"   Model: {STAGE_2_CONFIG['model_name']}")
print(f"   Target device: {STAGE_2_CONFIG['device'].upper()}")
print("   This may take a few minutes on first run...\n")

start_load = time.time()

# Load model and move to appropriate device
sbert_model = SentenceTransformer(STAGE_2_CONFIG['model_name'], device=STAGE_2_CONFIG['device'])

load_time = time.time() - start_load

print(f"✅ SBERT model loaded successfully in {load_time:.2f} seconds!")
print(f"   Model device: {STAGE_2_CONFIG['device'].upper()}")
print(f"   Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Verify GPU usage if CUDA is available
if STAGE_2_CONFIG['device'] == 'cuda':
    print(f"   GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"   GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print("   🚀 GPU acceleration enabled for faster processing!")

In [ ]:
# ============================================================================
# GPU MEMORY MANAGEMENT UTILITIES (Optional)
# ============================================================================
# Run this cell if you encounter GPU memory issues

def clear_gpu_memory():
    """Clear GPU cache to free up memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("✅ GPU cache cleared")
        print(f"   GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
        print(f"   GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    else:
        print("ℹ️  No GPU detected - nothing to clear")

def get_gpu_memory_info():
    """Display current GPU memory usage"""
    if torch.cuda.is_available():
        print(f"📊 GPU Memory Status:")
        print(f"   Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
        print(f"   Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
        free_memory = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3
        print(f"   Free: {free_memory:.2f} GB")
    else:
        print("ℹ️  No GPU detected")

# Uncomment to check GPU memory status
# get_gpu_memory_info()

print("✅ GPU utilities loaded (use clear_gpu_memory() or get_gpu_memory_info() if needed)")

## Step 4: Define Helper Functions

In [ ]:
# ============================================================================
# HELPER FUNCTIONS FOR CONTEXTUAL INPUT CREATION
# ============================================================================

def create_contextual_input_w3(prev_sent_i_minus_1, main_sent, next_sent_i_plus_1):
    """
    Create contextual input with window size 3 (1 previous + main + 1 next).
    
    Window 3 uses positions: (i-1, i, i+1)
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens
    """
    parts = []
    
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    parts.append(str(main_sent).strip())
    
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    return " [SEP] ".join(parts)


def create_contextual_input_w5(prev_sent_i_minus_2, prev_sent_i_minus_1, main_sent, 
                                next_sent_i_plus_1, next_sent_i_plus_2):
    """
    Create contextual input with window size 5 (2 previous + main + 2 next).
    
    Window 5 uses positions: (i-2, i-1, i, i+1, i+2)
    - prev_sent_i_minus_2: sentence at position (i-2) = previous_sentence_1 in Stage 1
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    - next_sent_i_plus_2: sentence at position (i+2) = next_sentence_2 in Stage 1
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens
    """
    parts = []
    
    if prev_sent_i_minus_2 and str(prev_sent_i_minus_2).strip():
        parts.append(str(prev_sent_i_minus_2).strip())
    
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    parts.append(str(main_sent).strip())
    
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    if next_sent_i_plus_2 and str(next_sent_i_plus_2).strip():
        parts.append(str(next_sent_i_plus_2).strip())
    
    return " [SEP] ".join(parts)


print("✅ Helper functions defined:")
print("   - create_contextual_input_w3() for 3-sentence window")
print("   - create_contextual_input_w5() for 5-sentence window")

## Step 5: Process Stage 1 Files One by One

In [ ]:
# ============================================================================
# MAIN PROCESSING FUNCTION - PROCESS SINGLE FILE
# ============================================================================

def process_single_file_stage2(input_file_path, file_number, total_files):
    """
    Process a single Stage 1 file and create context-aware embeddings with two window sizes.
    
    Args:
        input_file_path (Path): Path to the Stage 1 CSV file
        file_number (int): File number (for tracking)
        total_files (int): Total number of files
    
    Returns:
        tuple: (file_number, sentence_count, output_file_path, success, elapsed_time)
    """
    try:
        start_time = time.time()
        
        print(f"\n{'='*80}")
        print(f"Processing File {file_number}/{total_files}: {input_file_path.name}")
        print(f"{'='*80}")
        
        # Read the Stage 1 CSV file
        print("🔄 Loading CSV file...")
        df = pd.read_csv(input_file_path)
        print(f"   ✓ Loaded {len(df):,} sentences")
        
        # Validate required columns (updated for window size 5 structure)
        required_cols = ['sentence_id', 'article_id', 'date', 'source', 
                        'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 
                        'next_sentence_1', 'next_sentence_2']
        
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"⚠️  Skipping {input_file_path.name}: Missing columns {missing_cols}")
            return (file_number, 0, None, False, 0)
        
        # Replace NaN with empty strings
        df = df.fillna("")
        
        # Create contextual inputs for both window sizes
        print("\n🔄 Creating contextual inputs...")
        contextual_inputs_w3 = []
        contextual_inputs_w5 = []
        
        for idx, row in df.iterrows():
            # Window 3: Use positions (i-1, i, i+1)
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            input_w3 = create_contextual_input_w3(
                row['previous_sentence_2'],  # i-1
                row['main_sentence'],         # i
                row['next_sentence_1']        # i+1
            )
            contextual_inputs_w3.append(input_w3)
            
            # Window 5: Use positions (i-2, i-1, i, i+1, i+2)
            # - i-2 = previous_sentence_1
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            # - i+2 = next_sentence_2
            input_w5 = create_contextual_input_w5(
                row['previous_sentence_1'],  # i-2
                row['previous_sentence_2'],  # i-1
                row['main_sentence'],         # i
                row['next_sentence_1'],       # i+1
                row['next_sentence_2']        # i+2
            )
            contextual_inputs_w5.append(input_w5)
        
        print(f"   ✓ Created {len(contextual_inputs_w3):,} w3 contextual inputs")
        print(f"   ✓ Created {len(contextual_inputs_w5):,} w5 contextual inputs")
        
        # Generate embeddings in batches for efficiency
        batch_size = STAGE_2_CONFIG['batch_size']
        
        # Generate w3 embeddings
        print(f"\n🔄 Generating w3 embeddings (batch size: {batch_size})...")
        all_embeddings_w3 = []
        for i in range(0, len(contextual_inputs_w3), batch_size):
            batch = contextual_inputs_w3[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w3.extend(batch_embeddings)
            
            # Progress update every 20 batches
            if (i // batch_size) % 20 == 0:
                progress = min(i + batch_size, len(contextual_inputs_w3))
                print(f"   Progress: {progress:,}/{len(contextual_inputs_w3):,} sentences")
        
        print(f"   ✓ Generated {len(all_embeddings_w3):,} w3 embeddings")
        
        # Generate w5 embeddings
        print(f"\n🔄 Generating w5 embeddings (batch size: {batch_size})...")
        all_embeddings_w5 = []
        for i in range(0, len(contextual_inputs_w5), batch_size):
            batch = contextual_inputs_w5[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w5.extend(batch_embeddings)
            
            # Progress update every 20 batches
            if (i // batch_size) % 20 == 0:
                progress = min(i + batch_size, len(contextual_inputs_w5))
                print(f"   Progress: {progress:,}/{len(contextual_inputs_w5):,} sentences")
        
        print(f"   ✓ Generated {len(all_embeddings_w5):,} w5 embeddings")
        
        # Convert embeddings lists to numpy arrays
        embeddings_array_w3 = np.array(all_embeddings_w3)
        embeddings_array_w5 = np.array(all_embeddings_w5)
        
        # Add embeddings as new columns (store as comma-separated strings for CSV)
        print("\n🔄 Adding embeddings to dataframe...")
        df['w3_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w3]
        df['w5_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w5]
        print("   ✓ Embeddings added as columns")
        
        # Define output file path (preserve original filename structure)
        # Convert Data_s1_X.csv to Data_s2_X.csv
        output_filename = input_file_path.name.replace('Data_s1_', 'Data_s2_')
        if output_filename == input_file_path.name:  # If no replacement happened
            output_filename = f"Data_s2_{file_number}.csv"
        
        output_file = STAGE_2_OUTPUT / output_filename
        
        # Save to CSV
        print(f"\n🔄 Saving to: {output_file.name}...")
        df.to_csv(output_file, index=False)
        print("   ✓ File saved successfully")
        
        elapsed_time = time.time() - start_time
        
        print(f"\n✅ File {file_number}/{total_files} completed in {elapsed_time:.2f} seconds")
        print(f"   Sentences processed: {len(df):,}")
        print(f"   Output: {output_file.name}")
        
        return (file_number, len(df), output_file, True, elapsed_time)
        
    except Exception as e:
        print(f"\n❌ Error processing {input_file_path.name}: {e}")
        import traceback
        traceback.print_exc()
        return (file_number, 0, None, False, 0)


print("✅ Main processing function defined!")

In [ ]:
# ============================================================================
# STAGE 2: MAIN EXECUTION - PROCESS ALL FILES
# ============================================================================

print("=" * 100)
print("STAGE 2: SBERT EMBEDDINGS GENERATION (w3 + w5 WINDOWS)")
print("=" * 100)

# Get all Stage 1 output files
stage1_files = sorted(STAGE_1_INPUT.glob('*.csv'))
total_files = len(stage1_files)

if total_files == 0:
    print("\n❌ No CSV files found in Stage 1 folder!")
    print(f"   Please check: {STAGE_1_INPUT}")
else:
    print(f"\n📊 Processing Overview:")
    print(f"   Input folder: {STAGE_1_INPUT}")
    print(f"   Output folder: {STAGE_2_OUTPUT}")
    print(f"   Total files to process: {total_files}")
    print(f"   Model: {STAGE_2_CONFIG['model_name']}")
    print(f"   Batch size: {STAGE_2_CONFIG['batch_size']}")
    print(f"   Device: {STAGE_2_CONFIG['device'].upper()}")
    print("\n" + "=" * 100)
    
    # Process files sequentially (one by one)
    overall_start_time = time.time()
    results = []
    total_sentences_processed = 0
    successful_files = 0
    failed_files = 0
    
    for file_num, input_file in enumerate(stage1_files, start=1):
        result = process_single_file_stage2(input_file, file_num, total_files)
        results.append(result)
        
        file_number, sentence_count, output_file, success, elapsed_time = result
        
        if success:
            successful_files += 1
            total_sentences_processed += sentence_count
        else:
            failed_files += 1
    
    overall_elapsed_time = time.time() - overall_start_time
    
    # Print final summary
    print("\n" + "=" * 100)
    print("STAGE 2 PROCESSING COMPLETE")
    print("=" * 100)
    
    print(f"\n📊 Final Statistics:")
    print(f"   Total files processed: {total_files}")
    print(f"   Successful: {successful_files}")
    print(f"   Failed: {failed_files}")
    print(f"   Total sentences processed: {total_sentences_processed:,}")
    print(f"   Total processing time: {overall_elapsed_time:.2f} seconds ({overall_elapsed_time/60:.2f} minutes)")
    
    if total_sentences_processed > 0:
        avg_time_per_sentence = overall_elapsed_time / total_sentences_processed
        print(f"   Average time per sentence: {avg_time_per_sentence:.4f} seconds")
    
    print(f"\n✅ All Stage 2 files saved to: {STAGE_2_OUTPUT}")
    
    # List output files
    output_files = sorted(STAGE_2_OUTPUT.glob('*.csv'))
    print(f"\n📁 Output files ({len(output_files)} total):")
    for i, out_file in enumerate(output_files[:10], 1):  # Show first 10
        file_size_mb = out_file.stat().st_size / (1024 * 1024)
        print(f"   {i}. {out_file.name} ({file_size_mb:.2f} MB)")
    
    if len(output_files) > 10:
        print(f"   ... and {len(output_files) - 10} more files")
    
    print("\n🎉 Stage 2 processing completed successfully!")

## Step 6: Verification - Display Sample Results

In [ ]:
# ============================================================================
# VERIFICATION: DISPLAY SAMPLE RESULTS
# ============================================================================

print("=" * 100)
print("VERIFICATION: SAMPLE RESULTS FROM STAGE 2")
print("=" * 100)

# Load first output file for verification
output_files = sorted(STAGE_2_OUTPUT.glob('*.csv'))

if len(output_files) > 0:
    sample_file = output_files[0]
    print(f"\n📋 Loading sample file: {sample_file.name}\n")
    
    sample_df = pd.read_csv(sample_file)
    
    print(f"Total sentences in file: {len(sample_df):,}")
    print(f"Columns: {list(sample_df.columns)}")
    print("\n" + "=" * 100)
    print("Sample Sentences with Embeddings (first 2 rows)")
    print("=" * 100)
    
    for i in range(min(2, len(sample_df))):
        row = sample_df.iloc[i]
        print(f"\n🔹 Sentence {i+1}:")
        print(f"   Sentence ID: {row['sentence_id']}")
        print(f"   Article ID: {row['article_id']}")
        print(f"   Date: {row['date']}")
        print(f"   Source: {row['source']}")
        
        print(f"\n   5-Sentence Window Context:")
        print(f"   └─ Previous 2 (i-2): {str(row['previous_sentence_1'])[:100]}...")
        print(f"   └─ Previous 1 (i-1): {str(row['previous_sentence_2'])[:100]}...")
        print(f"   └─ Main (i):         {str(row['main_sentence'])[:100]}...")
        print(f"   └─ Next 1 (i+1):     {str(row['next_sentence_1'])[:100]}...")
        print(f"   └─ Next 2 (i+2):     {str(row['next_sentence_2'])[:100]}...")
        
        # Show embedding info (not full embedding to save space)
        w3_emb = row['w3_embedding'].split(',')[:5]
        w5_emb = row['w5_embedding'].split(',')[:5]
        w3_emb_len = len(row['w3_embedding'].split(','))
        w5_emb_len = len(row['w5_embedding'].split(','))
        
        print(f"\n   Embeddings Generated:")
        print(f"   └─ w3 embedding: {w3_emb_len} dimensions, first 5: [{', '.join(w3_emb)}...]")
        print(f"   └─ w5 embedding: {w5_emb_len} dimensions, first 5: [{', '.join(w5_emb)}...]")
        print("\n" + "-" * 100)
    
    print("\n✅ Verification complete! Embeddings look good.")
    print("\n💡 Next Steps:")
    print("   - Use Stage 2 data for downstream analysis")
    print("   - Both w3 and w5 embeddings are available in each file")
    print("   - Files are ready for narrative shift detection or classification tasks")
    
else:
    print("\n⚠️  No output files found. Please run Stage 2 processing first.")